# 06a — M2 (statistical global): COMBINED detector (PTB-XL + Ningbo)

Clean rebuild of legacy 08a, English, with the **unified parsimony selection (1-SE rule)**: among depth-2 configs under the gap cap, the best OOF AP defines a bootstrap IC95 lower bound, and K is the **smallest value statistically tied** with it (same IC principle as K_auto). Selection never uses fold 9. Representation = 100% event-free. Judge: AP. Fold 10 untouched.

**Inputs:** `metadata_combined.csv`, `signal_loading`, `m2_features.csv` (shared, built here).
**Outputs:** `models/M2_statistical/m2_*`, `reports/{figures,metrics}/M2_combined_*`.

> Legacy 08a kept as reference. Narrative: see `JOURNAL.md`.

### Block 6a.0: Setup, paths, frozen filter, leads, integrity

In [ ]:
# M2 (statistical global) — COMBINED detector (PTB-XL + Ningbo). Clean rebuild of legacy 08a, English,
# with the UNIFIED selection rule (same as M1 05a/b/c: max OOF AP among depth-2 configs under an
# absolute gap cap <= 0.30; selection never uses the held-out fold 9). Representation = 100% event-free
# (distribution stats / spectrum / autocorrelation), no delineation, no peak detection. Judge: AP.
# Fold 10 UNTOUCHED. (Legacy 08a kept as reference.)
import os, sys, json
import numpy as np
import pandas as pd

ROOT      = r'.'
PROCESSED = os.path.join(ROOT, 'data', 'processed')
MODELS    = os.path.join(ROOT, 'models', 'M2_statistical')
SRC       = os.path.join(ROOT, 'src')
FIG       = os.path.join(ROOT, 'reports', 'figures')
METRICS   = os.path.join(ROOT, 'reports', 'metrics')
for d in (MODELS, FIG, METRICS): os.makedirs(d, exist_ok=True)
sys.path.insert(0, SRC)

# Canonical signal loader (single source of truth for signal loading)
from signal_loading import load_signal, LEADS_CANONICAL
print("signal_loading OK - canonical lead order:", LEADS_CANONICAL)

# Frozen filter (decided in 06e) - read, never re-decided
with open(os.path.join(PROCESSED, 'filter_config.json'), encoding='utf-8') as f:
    FCFG = json.load(f)['filter_FINAL']
assert (FCFG['low'], FCFG['high'], FCFG['order']) == (0.5, 40, 4), "Unexpected filter!"
FS = FCFG['fs']
print(f"Frozen filter: {FCFG['type']} order {FCFG['order']}, {FCFG['low']}-{FCFG['high']} Hz, {FCFG['phase']}")

# M2 = WIDE: all 12 leads. The FDR gate will decide.
LEADS_M2 = list(LEADS_CANONICAL)
LEAD_IDX = {L: LEADS_CANONICAL.index(L) for L in LEADS_M2}
print(f"M2 leads: {len(LEADS_M2)} (all 12) -> {LEADS_M2}")

FEATURES_CSV = os.path.join(PROCESSED, 'm2_features.csv')   # built in 6a.1 (shared by 06a/b/c)
meta = pd.read_csv(os.path.join(PROCESSED, 'metadata_combined.csv'), dtype={'ecg_id': str})
n_wpw = int((meta.label == 1).sum())
print(f"\nmetadata_combined: {len(meta)} ECGs, {n_wpw} WPW")
print(f"WPW per fold: {meta[meta.label==1].groupby('fold').size().to_dict()}")
print(f"Train(1-8): {int(meta[(meta.fold.between(1,8))&(meta.label==1)].shape[0])} | "
      f"Val(9): {int(meta[(meta.fold==9)&(meta.label==1)].shape[0])} | "
      f"Test(10): {int(meta[(meta.fold==10)&(meta.label==1)].shape[0])} [UNTOUCHED]")
assert n_wpw == 142, "Expected 142 WPW in the combined set!"
print("\nBlock 6a.0 OK - context loaded, filter verified. Ready for 6a.1 (massive M2 extraction).")


### Block 6a.0b: Patient-leakage check (blocking assert)

In [ ]:
# Patient-leakage check across folds (blocking assert). A patient in 2 folds = leakage
# (the model would recognize the patient, not the disease).
pat_folds = meta.groupby('patient_id')['fold'].nunique()
leaky = pat_folds[pat_folds > 1]
print(f"Unique patients: {meta.patient_id.nunique()} | ECGs: {len(meta)}")
print(f"Patients in >1 fold: {len(leaky)}")
wpw_pat = meta[meta.label==1].patient_id.nunique()
print(f"WPW: {int((meta.label==1).sum())} ECGs across {wpw_pat} unique patients")
if len(leaky) > 0:
    print("\nLEAK DETAIL (first 10):")
    print(meta[meta.patient_id.isin(leaky.index)].groupby('patient_id')['fold']
          .apply(lambda s: sorted(s.unique())).head(10).to_string())
assert len(leaky) == 0, (f"PATIENT LEAK: {len(leaky)} patients in multiple folds. "
                         "Fix fold construction before continuing.")
print("\n[OK] No patient leakage across folds. Folds are clean.")


### Block 6a.1: Massive M2 statistical extraction (100% event-free)

In [ ]:
# Massive M2 statistical extraction (100% event-free). Families: temporal stats (+ higher-order,
# Teager-Kaiser, threshold crossings), Hjorth (+ 2nd/3rd derivatives, |signal|), spectral (bands +
# fine 2.5 Hz comb, roll-off, flux), autocorrelation (rhythm + lags + harmonics), nonlinear (time
# asymmetry), multiscale permutation entropy. ~120-130 feat/lead x 12 ~ 1450-1550 features.
# Steps: synthetic test -> 200-ECG timing (ETA) -> full run (RUN_FULL=True). Anti-rerun guard.
FORCE_REEXTRACT = False     # True to regenerate even if the CSV exists
RUN_FULL        = True      # set True AFTER checking the 200-ECG ETA

import warnings, time, contextlib, math
import numpy as np
from scipy.signal import butter, sosfiltfilt, welch
from scipy.stats import skew, kurtosis, iqr
from joblib import Parallel, delayed
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Frozen band-pass filter (identical to training and Flask)
SOS_BP = butter(FCFG['order'], [FCFG['low']/(FS/2), FCFG['high']/(FS/2)], btype='band', output='sos')
def bp(x): return sosfiltfilt(SOS_BP, np.asarray(x, dtype=np.float64))

# tqdm bridge for joblib
@contextlib.contextmanager
def tqdm_joblib(t):
    import joblib
    class _Cb(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *a, **k): t.update(n=self.batch_size); return super().__call__(*a, **k)
    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = _Cb
    try: yield t
    finally: joblib.parallel.BatchCompletionCallBack = old; t.close()

# Spectral bands (Hz). 15-40 = delta-sensitive band (cf. 6a.2).
BANDS = {
    'tot': (0.5, 40), 'vlf': (0.5, 3), 'lf': (3, 8), 'mf': (8, 15),
    'delta_hf': (15, 25), 'hf': (25, 40), 'd1540': (15, 40),
}

# ===========================================================================
# FEATURE FAMILIES (all 100% peak-detection-free)
# ===========================================================================
def f_temporal(x):
    """Amplitude-distribution statistics (no notion of heartbeat)."""
    o = {}
    o['mean']   = float(np.mean(x));      o['std']  = float(np.std(x))
    o['var']    = float(np.var(x));       o['rms']  = float(np.sqrt(np.mean(x**2)))
    o['mad']    = float(np.median(np.abs(x - np.median(x))))
    o['iqr']    = float(iqr(x));          o['rng']  = float(np.ptp(x))
    o['min']    = float(np.min(x));       o['max']  = float(np.max(x))
    o['p05']    = float(np.percentile(x, 5));  o['p25'] = float(np.percentile(x, 25))
    o['p75']    = float(np.percentile(x, 75)); o['p95'] = float(np.percentile(x, 95))
    o['skew']   = float(skew(x));         o['kurt'] = float(kurtosis(x))
    o['absmean']= float(np.mean(np.abs(x)))
    o['absmax'] = float(np.max(np.abs(x)))
    o['crest']  = float(np.max(np.abs(x)) / (o['rms'] + 1e-9))
    o['shape']  = float(o['rms'] / (o['absmean'] + 1e-9))
    o['impulse']= float(np.max(np.abs(x)) / (o['absmean'] + 1e-9))
    xc = x - np.median(x)
    o['zcr']    = float(np.mean(np.diff(np.sign(xc)) != 0))
    d1 = np.diff(x) * FS
    o['dabs_mean'] = float(np.mean(np.abs(d1)))
    o['dabs_p95']  = float(np.percentile(np.abs(d1), 95))
    o['dabs_max']  = float(np.max(np.abs(d1)))
    o['d_std']     = float(np.std(d1))
    return o

def f_temporal_plus(x):
    o = {}
    s = np.std(x) + 1e-12
    z = (x - np.mean(x)) / s
    o['mom5'] = float(np.mean(z**5)); o['mom6'] = float(np.mean(z**6))
    p = np.percentile(x, [10,25,50,75,90])
    o['pratio_9010'] = float((p[4]-p[0]) / (abs(p[2])+1e-9))
    o['qcoef']       = float((p[3]-p[1]) / (p[3]+p[1]+1e-9))
    o['p90_p50']     = float(p[4] / (abs(p[2])+1e-9))
    tk = x[1:-1]**2 - x[:-2]*x[2:]              # Teager-Kaiser energy
    o['tk_mean'] = float(np.mean(np.abs(tk)))
    o['tk_std']  = float(np.std(tk))
    o['tk_p95']  = float(np.percentile(np.abs(tk), 95))
    for k in (1, 2, 3):
        thr = k * s
        o[f'frac_gt_{k}sd'] = float(np.mean(np.abs(x - np.mean(x)) > thr))
        o[f'ncross_{k}sd']  = int(np.sum(np.diff((np.abs(x-np.mean(x)) > thr).astype(int)) == 1))
    return o

def f_nonlinear(x):
    """Time asymmetry (time-reversibility): steep fronts vs gentle slopes (delta wave)."""
    o = {}
    s = np.std(x) + 1e-12
    for k in (2, 4, 8):                          # several time lags
        d = x[k:] - x[:-k]
        d3 = np.mean(d**3); d2 = np.mean(d**2)
        o[f'trev_{k}'] = float(d3 / (d2**1.5 + 1e-12))   # normalized asymmetry
        o[f'dasym_{k}'] = float((np.mean(d[d>0]**2) if (d>0).any() else 0.0)
                               - (np.mean(d[d<0]**2) if (d<0).any() else 0.0)) / (s**2)
    d1 = np.diff(x)
    up = d1[d1>0]; dn = d1[d1<0]
    o['slope_updown_ratio'] = float((np.mean(up) if up.size else 0.0) /
                                    (abs(np.mean(dn)) + 1e-12))
    o['slope_up_frac']      = float(np.mean(d1 > 0))
    return o

def f_hjorth(x):
    """Activity / mobility / complexity (derivative-based descriptors)."""
    d1 = np.diff(x); d2 = np.diff(d1)
    v0 = np.var(x) + 1e-12; v1 = np.var(d1) + 1e-12; v2 = np.var(d2) + 1e-12
    mob = np.sqrt(v1 / v0)
    return {'hj_activity': float(v0),
            'hj_mobility': float(mob),
            'hj_complexity': float(np.sqrt(v2 / v1) / (mob + 1e-12))}

def f_hjorth_plus(x):
    d1 = np.diff(x); d2 = np.diff(d1); d3 = np.diff(d2)
    v1 = np.var(d1)+1e-12; v2 = np.var(d2)+1e-12; v3 = np.var(d3)+1e-12
    mob2 = np.sqrt(v2 / v1)
    ax = np.abs(x - np.mean(x))
    da = np.diff(ax); va0 = np.var(ax)+1e-12; va1 = np.var(da)+1e-12
    return {'hj2_mobility': float(mob2),
            'hj2_complexity': float(np.sqrt(v3/v2) / (mob2+1e-12)),
            'hj_abs_mobility': float(np.sqrt(va1/va0))}

def f_spectral(x):
    """Global spectrum (Welch). Band energies + spectral descriptors."""
    o = {}
    nper = min(1024, len(x))
    freqs, psd = welch(x, fs=FS, nperseg=nper)
    tot = np.trapezoid(psd, freqs) + 1e-12
    for name, (lo, hi) in BANDS.items():
        m = (freqs >= lo) & (freqs < hi)
        e = float(np.trapezoid(psd[m], freqs[m])) if m.any() else 0.0
        o[f'sp_e_{name}']   = e
        o[f'sp_rel_{name}'] = float(e / tot)
    o['sp_ratio_hf_lf']   = o['sp_e_hf']      / (o['sp_e_lf'] + 1e-12)
    o['sp_ratio_d1540']   = o['sp_e_d1540']   / tot
    o['sp_ratio_dhf_mf']  = o['sp_e_delta_hf']/ (o['sp_e_mf'] + 1e-12)
    o['sp_log_d1540']     = float(np.log(o['sp_e_d1540'] + 1e-12))
    o['sp_log_lf']        = float(np.log(o['sp_e_lf'] + 1e-12))
    o['sp_ratio_hf_mf']   = o['sp_e_hf']      / (o['sp_e_mf'] + 1e-12)
    o['sp_ratio_dhf_lf']  = o['sp_e_delta_hf']/ (o['sp_e_lf'] + 1e-12)
    psn = psd / (psd.sum() + 1e-12)
    o['sp_centroid'] = float(np.sum(freqs * psn))
    o['sp_bw']       = float(np.sqrt(np.sum(((freqs - o['sp_centroid'])**2) * psn)))
    o['sp_entropy']  = float(-np.sum(psn * np.log(psn + 1e-12)))
    o['sp_flatness'] = float(np.exp(np.mean(np.log(psd + 1e-12))) / (np.mean(psd) + 1e-12))
    csum = np.cumsum(psd) / (psd.sum() + 1e-12)
    o['sp_median']   = float(freqs[np.searchsorted(csum, 0.5)])
    o['sp_edge95']   = float(freqs[min(np.searchsorted(csum, 0.95), len(freqs)-1)])
    o['sp_peak']     = float(freqs[int(np.argmax(psd))])
    o['sp_peak_pow'] = float(psd.max() / (np.mean(psd) + 1e-12))
    return o

def f_spectral_fine(x):
    o = {}
    nper = min(1024, len(x))
    freqs, psd = welch(x, fs=FS, nperseg=nper)
    tot = np.trapezoid(psd, freqs) + 1e-12
    edges = np.arange(0, 40.0001, 2.5)         # narrow 2.5 Hz comb
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (freqs >= lo) & (freqs < hi)
        o[f'spf_{lo:.0f}_{hi:.0f}'] = float(np.trapezoid(psd[m], freqs[m]) / tot) if m.any() else 0.0
    csum = np.cumsum(psd) / (psd.sum() + 1e-12)
    for q in (0.25, 0.50, 0.75, 0.90):
        o[f'sp_roll{int(q*100)}'] = float(freqs[min(np.searchsorted(csum, q), len(freqs)-1)])
    h = len(x)//2
    _, p1 = welch(x[:h], fs=FS, nperseg=min(512, h))
    _, p2 = welch(x[h:], fs=FS, nperseg=min(512, len(x)-h))
    n = min(len(p1), len(p2))
    p1n = p1[:n]/(p1[:n].sum()+1e-12); p2n = p2[:n]/(p2[:n].sum()+1e-12)
    o['sp_flux'] = float(np.sqrt(np.sum((p1n - p2n)**2)))
    return o

def f_autocorr(x):
    """Rhythm and regularity via AUTOCORRELATION (never peak detection)."""
    o = {}
    xz = x - np.mean(x)
    ac = np.correlate(xz, xz, mode='full')[len(xz)-1:]
    ac = ac / (ac[0] + 1e-12)
    lo = int(FS * 60 / 220); hi = int(FS * 60 / 30); hi = min(hi, len(ac) - 2)
    if hi > lo + 2:
        seg = ac[lo:hi]; dac = np.diff(seg)
        peaks = np.where((np.hstack([dac, 0]) < 0) & (np.hstack([0, dac]) > 0))[0]
        if peaks.size:
            pk = peaks[int(np.argmax(seg[peaks]))]; lag = lo + pk
            o['ac_peak_height'] = float(seg[pk])
            o['ac_peak_lag_ms'] = float(lag / FS * 1000)
            o['ac_hr_bpm']      = float(60 * FS / lag)
            half = seg[pk] / 2; l = pk; r = pk
            while l > 0 and seg[l] > half: l -= 1
            while r < len(seg)-1 and seg[r] > half: r += 1
            o['ac_peak_width_ms'] = float((r - l) / FS * 1000)
        else:
            o.update(ac_peak_height=np.nan, ac_peak_lag_ms=np.nan, ac_hr_bpm=np.nan, ac_peak_width_ms=np.nan)
    else:
        o.update(ac_peak_height=np.nan, ac_peak_lag_ms=np.nan, ac_hr_bpm=np.nan, ac_peak_width_ms=np.nan)
    a = int(0.05*FS); b = min(int(0.5*FS), len(ac)-1)
    if b > a + 2:
        seg2 = np.abs(ac[a:b])
        o['ac_decay'] = float(np.polyfit(np.arange(len(seg2)), seg2, 1)[0])
        o['ac_mean_abs'] = float(np.mean(seg2))
    else:
        o.update(ac_decay=np.nan, ac_mean_abs=np.nan)
    return o

def f_autocorr_plus(x):
    o = {}
    xz = x - np.mean(x)
    ac = np.correlate(xz, xz, mode='full')[len(xz)-1:]
    ac = ac / (ac[0] + 1e-12)
    for ms in (10, 20, 50, 100, 200, 300):
        lag = int(ms/1000*FS)
        o[f'ac_lag{ms}'] = float(ac[lag]) if lag < len(ac) else np.nan
    zc = np.where(np.diff(np.sign(ac)) != 0)[0]
    o['ac_first_zero_ms'] = float(zc[0]/FS*1000) if zc.size else np.nan
    lo = int(FS*60/220); hi = min(int(FS*60/30)*2, len(ac)-2)
    if hi > lo + 4:
        seg = ac[lo:hi]; dac = np.diff(seg)
        pk = np.where((np.hstack([dac,0])<0) & (np.hstack([0,dac])>0))[0]
        pk = pk[np.argsort(seg[pk])[::-1]] if pk.size else pk
        o['ac_2nd_1st_ratio'] = float(seg[pk[1]] / (seg[pk[0]]+1e-9)) if pk.size >= 2 else np.nan
    else:
        o['ac_2nd_1st_ratio'] = np.nan
    return o

def _perm_ent(x, m, tau):
    n = len(x) - (m - 1) * tau
    if n <= 1: return np.nan
    patterns = {}
    idx = np.arange(0, n, max(1, n // 4000))
    for i in idx:
        vec = x[i:i + m*tau:tau]
        key = tuple(np.argsort(vec))
        patterns[key] = patterns.get(key, 0) + 1
    counts = np.array(list(patterns.values()), float)
    p = counts / counts.sum()
    return float(-np.sum(p * np.log(p)) / np.log(math.factorial(m)))

def f_perm_multiscale(x):
    """Permutation entropy at several orders/scales (multi-resolution complexity)."""
    return {
        'perm_ent':       _perm_ent(x, 3, 4),   # reference
        'perm_ent_m4':    _perm_ent(x, 4, 4),
        'perm_ent_tau2':  _perm_ent(x, 3, 2),
        'perm_ent_tau8':  _perm_ent(x, 3, 8),
    }

def extract_lead(x):
    """All families on one lead. ~120-130 features."""
    o = {}
    o.update(f_temporal(x));      o.update(f_temporal_plus(x))
    o.update(f_nonlinear(x))
    o.update(f_hjorth(x));        o.update(f_hjorth_plus(x))
    o.update(f_spectral(x));      o.update(f_spectral_fine(x))
    o.update(f_autocorr(x));      o.update(f_autocorr_plus(x))
    o.update(f_perm_multiscale(x))
    return o

def process_one(m):
    warnings.filterwarnings('ignore')
    row = {'ecg_id': m['ecg_id'], 'patient_id': m['patient_id'], 'label': m['label'],
           'fold': m['fold'], 'source': m['source'], 'extraction_failed': 0}
    try:
        sig = load_signal(m['ecg_id'], m['source'])
        for L in LEADS_M2:
            xf = bp(sig[:, LEAD_IDX[L]])
            for k, v in extract_lead(xf).items():
                row[f'{k}_{L}'] = v
    except Exception:
        row['extraction_failed'] = 1
    return row

# (1) SYNTHETIC TEST
print("Synthetic test...")
t = np.arange(5000) / FS
synth = (np.sin(2*np.pi*1.2*t) + 0.3*np.random.randn(5000))
fe = extract_lead(bp(synth))
print(f"  OK - {len(fe)} features/lead, {len(fe)*len(LEADS_M2)} features/ECG expected.")
assert all(np.isfinite(v) or (isinstance(v,float) and np.isnan(v)) for v in fe.values()), "non-numeric value!"

# Anti-rerun guard
if os.path.exists(FEATURES_CSV) and not FORCE_REEXTRACT:
    print(f"\n{FEATURES_CSV} exists -> extraction SKIPPED (set FORCE_REEXTRACT=True to regenerate).")
else:
    recs = meta.to_dict('records')
    # (2) TIMING on 200 ECGs -> ETA
    sample = recs[:200]
    t0 = time.time()
    with tqdm_joblib(tqdm(total=len(sample), desc='Timing 200', unit='ecg')):
        _ = Parallel(n_jobs=10, backend='loky')(delayed(process_one)(m) for m in sample)
    dt = time.time() - t0
    eta_min = dt / len(sample) * len(recs) / 60
    print(f"\n  200 ECGs in {dt:.1f}s -> full-run ETA ({len(recs)} ECGs) ~ {eta_min:.0f} min "
          f"({eta_min/60:.1f} h) on 10 cores.")
    if not RUN_FULL:
        print("\n  >>> RUN_FULL=False: full run NOT launched. Check the ETA, set RUN_FULL=True and re-run this cell.")
    else:
        # (3) FULL RUN
        t0 = time.time()
        with tqdm_joblib(tqdm(total=len(recs), desc='M2 massive extraction', unit='ecg')):
            rows = Parallel(n_jobs=10, backend='loky')(delayed(process_one)(m) for m in recs)
        df_ = pd.DataFrame(rows); df_.to_csv(FEATURES_CSV, index=False)
        meta_cols = ['ecg_id','patient_id','label','fold','source','extraction_failed']
        nfeat = df_.shape[1] - len(meta_cols)
        print(f"\nDone in {(time.time()-t0)/60:.1f} min -> {FEATURES_CSV}")
        print(f"Shape: {df_.shape}  ({nfeat} features)")
        for s in df_.source.unique():
            sub = df_[df_.source == s]
            print(f"  {s}: failed WPW {sub[sub.label==1].extraction_failed.mean()*100:.1f}% | "
                  f"non-WPW {sub[sub.label==0].extraction_failed.mean()*100:.1f}%")
        nan_rate = df_.drop(columns=meta_cols).isna().mean() * 100
        print(f"\nFeatures with >40% NaN: {(nan_rate>40).sum()} / {len(nan_rate)}")
        print(nan_rate.sort_values(ascending=False).head(8).round(1).to_string())


### Block 6a.2: Feature selection (FDR gate + cross-dataset + dedup)

In [ ]:
# Feature selection: FDR gate + cross-dataset coherence + fast de-duplication.
# Gate on folds 1-8: |Cohen's d|>0.3 AND p_FDR<0.05 (Benjamini-Hochberg) AND IC95 bootstrap of d
# excludes 0 AND cross-dataset coherence (separates in PTB AND Ningbo). Then Spearman>0.9 dedup
# (fast variant: pre-ranked columns + np.corrcoef = Spearman).
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

df = pd.read_csv(FEATURES_CSV, dtype={'ecg_id': str})
META = ['ecg_id','patient_id','label','fold','source','extraction_failed']
ALL_FEATS = [c for c in df.columns if c not in META]
tr = df[df.fold.between(1,8)]
print(f"Candidate features: {len(ALL_FEATS)} | train folds 1-8: {len(tr)} ECGs, {int((tr.label==1).sum())} WPW")

def cohens_d(a,b):
    a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return np.nan
    sp=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    return (a.mean()-b.mean())/sp if sp>0 else np.nan
def d_ci(a,b,n=1000,seed=42):
    rng=np.random.default_rng(seed); a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return (np.nan,np.nan)
    ds=[cohens_d(rng.choice(a,len(a),True),rng.choice(b,len(b),True)) for _ in range(n)]
    return tuple(np.nanpercentile(ds,[2.5,97.5]))

SEL_CSV = os.path.join(PROCESSED,'m2_combined_selection.csv')   # reused from legacy (identical logic)
if os.path.exists(SEL_CSV):
    res = pd.read_csv(SEL_CSV)
    print(f"[guard] {SEL_CSV} reloaded ({len(res)} features).")
else:
    w,n=tr[tr.label==1],tr[tr.label==0]
    ptb,nin=tr[tr.source=='ptbxl'],tr[tr.source=='ningbo']
    rows=[]
    for c in tqdm(ALL_FEATS, desc='Gate (d + p + IC + cross)', unit='feat'):
        a,b=w[c].values.astype(float),n[c].values.astype(float)
        d=cohens_d(a,b)
        try: _,p=mannwhitneyu(a[~np.isnan(a)],b[~np.isnan(b)],alternative='two-sided')
        except Exception: p=np.nan
        lo,hi=d_ci(a,b)
        dp=cohens_d(ptb[ptb.label==1][c].values.astype(float),ptb[ptb.label==0][c].values.astype(float))
        dn=cohens_d(nin[nin.label==1][c].values.astype(float),nin[nin.label==0][c].values.astype(float))
        cross_ok=(np.isfinite(dp) and np.isfinite(dn) and np.sign(dp)==np.sign(dn) and abs(dp)>0.2 and abs(dn)>0.2)
        ci_ok=(np.isfinite(lo) and np.isfinite(hi) and (lo>0)==(hi>0))
        rows.append({'feature':c,'d':d,'d_ptb':dp,'d_nin':dn,'p_raw':p,'ci_excl0':ci_ok,'cross_ok':cross_ok})
    res=pd.DataFrame(rows); ok=res.p_raw.notna()
    res.loc[ok,'p_FDR']=multipletests(res.loc[ok,'p_raw'],method='fdr_bh')[1]
    res['gate']=(res.d.abs()>0.3)&(res.p_FDR<0.05)&res.ci_excl0&res.cross_ok
    res=res.sort_values('d',key=lambda s:s.abs(),ascending=False).reset_index(drop=True)
    res.to_csv(SEL_CSV,index=False)

passed=res[res.gate].feature.tolist()
print(f"Features tested: {len(ALL_FEATS)} | pass the strict gate: {len(passed)}")

# Spearman>0.9 de-duplication - FAST (pre-ranked columns + np.corrcoef = Spearman)
def dedup_fast(feats, cap=None):
    sub = tr[feats].rank().to_numpy()
    col_means = np.nanmean(sub, axis=0)
    inds = np.where(np.isnan(sub))
    sub[inds] = np.take(col_means, inds[1])
    C = np.abs(np.corrcoef(sub, rowvar=False))
    idx = {f:i for i,f in enumerate(feats)}; keep=[]
    for f in feats:
        i=idx[f]
        if all(C[i, idx[k]] <= 0.9 for k in keep): keep.append(f)
        if cap and len(keep)>=cap: break
    return keep
dedup_list = dedup_fast(passed)
print(f"After dedup>0.9: {len(dedup_list)} features\n")
pd.set_option('display.float_format',lambda x:f'{x:.3f}')
print(res[['feature','d','d_ptb','d_nin','p_FDR','cross_ok','gate']].head(20).to_string(index=False))


### Block 6a.2b: Histograms of top selected features

In [ ]:
# Histograms of the top-6 selected features (WPW vs non-WPW, folds 1-8) - visual separation check.
import matplotlib.pyplot as plt
top6 = dedup_list[:6]
fig,ax=plt.subplots(2,3,figsize=(15,7))
for j,f in enumerate(top6):
    a=ax[j//3,j%3]
    wv=tr[tr.label==1][f].dropna(); nv=tr[tr.label==0][f].dropna()
    a.hist(nv,bins=50,alpha=.5,density=True,label='non-WPW',color='#94a3b8')
    a.hist(wv,bins=30,alpha=.6,density=True,label='WPW',color='#dc2626')
    a.set_title(f"{f}\n(d={res.set_index('feature').loc[f,'d']:.2f})",fontsize=9); a.legend(fontsize=7)
plt.suptitle('M2 combined - top-6 selected features (folds 1-8)'); plt.tight_layout()
plt.savefig(os.path.join(FIG,'m2_combined_histograms.png'),dpi=130,bbox_inches='tight'); plt.show()


### Block 6a.3: AP-vs-k + fold-9 overfit guard + provisional auto-K

In [ ]:
# AP vs k (1..k_max) + fold-9 overfit probe + provisional auto-K. Produces ONLY the OOF folds-1-8 curve
# (IC95 bootstrap) + a fold-9 probe + a provisional K_auto. Guard: reload m2_ap_vs_k.csv (~2h20 else).
# Final K & config are decided in 6a.3d (2D K x config sweep under the gap constraint).
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier
from tqdm import tqdm
import matplotlib.pyplot as plt, time

RK_CSV = os.path.join(PROCESSED, 'm2_ap_vs_k.csv')   # reused from legacy (identical logic)

d8 = df[df.fold.between(1,8)].reset_index(drop=True)
y, folds = d8.label.values, d8.fold.values
FOLD_MASKS = [(folds!=h, folds==h) for h in sorted(np.unique(folds))]
f9 = df[df.fold==9].reset_index(drop=True); yf9 = f9.label.values
spw8 = (y==0).sum()/max((y==1).sum(),1)

def make_xgb(spw=None, **kw):
    if spw is None: spw=spw8
    p=dict(n_estimators=100,max_depth=2,learning_rate=0.05,subsample=0.8,
           colsample_bytree=0.8,reg_lambda=2.0,min_child_weight=3,scale_pos_weight=spw,
           eval_metric='aucpr',tree_method='hist',random_state=42,n_jobs=10)
    p.update(kw); return XGBClassifier(**p)

def oof_xgb(feats, **kw):
    Xall=d8[feats]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        spw=(y[trm]==0).sum()/max((y[trm]==1).sum(),1)
        oof[vam]=make_xgb(spw,**kw).fit(Xall[trm],y[trm]).predict_proba(Xall[vam])[:,1]
    return oof

def make_boot_idx(npos,nneg,seed=42,n=1000):
    rng=np.random.default_rng(seed); pos=np.arange(npos); neg=np.arange(npos,npos+nneg)
    return np.stack([np.concatenate([rng.choice(pos,npos,True),rng.choice(neg,nneg,True)]) for _ in range(n)])
def ap_boot(yy,pp,bi):
    o=np.argsort(-yy); yy2,pp2=yy[o],pp[o]
    a=np.fromiter((average_precision_score(yy2[ix],pp2[ix]) for ix in bi),float,len(bi))
    return np.median(a),np.percentile(a,2.5),np.percentile(a,97.5)

if os.path.exists(RK_CSV):
    rk=pd.read_csv(RK_CSV); print(f"[guard] {RK_CSV} reloaded ({len(rk)} k).")
else:
    PROBE9=set(range(10,len(dedup_list)+1,10))
    ks=list(range(1,len(dedup_list)+1)); rows=[]; boot_cache={}; t0=time.time()
    for k in tqdm(ks, desc='AP vs k', unit='k'):
        feats=dedup_list[:k]
        oof=oof_xgb(feats); m=~np.isnan(oof); ym=y[m]; pm=oof[m]
        npos=int(ym.sum()); nneg=int((ym==0).sum()); key=(npos,nneg)
        if key not in boot_cache: boot_cache[key]=make_boot_idx(npos,nneg)
        med,lo,hi=ap_boot(ym,pm,boot_cache[key])
        r={'k':k,'AP_med':med,'IC_lo':lo,'IC_hi':hi,'AP_f9':np.nan,'AUC_f9':np.nan}
        if k in PROBE9:
            mdl=make_xgb().fit(d8[feats],y); p9=mdl.predict_proba(f9[feats])[:,1]
            r['AP_f9']=average_precision_score(yf9,p9); r['AUC_f9']=roc_auc_score(yf9,p9)
        rows.append(r)
        if k%25==0: pd.DataFrame(rows).to_csv(RK_CSV,index=False)
    rk=pd.DataFrame(rows); rk.to_csv(RK_CSV,index=False)
    print(f"Curve computed in {(time.time()-t0)/60:.0f} min.")

# Plot
fig,ax=plt.subplots(figsize=(14,6))
ax.errorbar(rk.k,rk.AP_med,yerr=[rk.AP_med-rk.IC_lo,rk.IC_hi-rk.AP_med],fmt='o-',capsize=2,color='#2563eb',ms=2.5,lw=.8,label='AP OOF folds 1-8 (IC95)')
f9p=rk.dropna(subset=['AP_f9'])
ax.plot(f9p.k,f9p.AP_f9,'s-',color='#dc2626',ms=6,label='AP fold 9 (overfit guard)')
ax.axhline(0.583,ls='--',color='gray',label='M6 ref (0.583)')
ax.axhline(0.617,ls=':',color='#16a34a',label='M1 combined (0.617)')
ax.set(xlabel='k (number of features)',ylabel='AP'); ax.legend(); ax.grid(alpha=.3)
ax.set_title('M2 combined - AP vs k: OOF + fold-9 validation')
plt.savefig(os.path.join(FIG,'m2_combined_ap_vs_k.png'),dpi=150,bbox_inches='tight'); plt.show()

# Provisional auto-K (on OOF, IC_lo of max) + fold-9 overfit check
kbest=rk.loc[rk.AP_med.idxmax()]; thr=kbest.IC_lo
K_auto=int(rk[rk.AP_med>=thr].k.min())
f9_low =f9p[f9p.k<=K_auto].AP_f9.mean(); f9_high=f9p[f9p.k>K_auto].AP_f9.mean() if (f9p.k>K_auto).any() else np.nan
print(f"Max OOF AP at k={int(kbest.k)} (AP={kbest.AP_med:.4f}). Plateau (>= IC_lo of max) from k={K_auto}.")
print(f"fold9 check: AP mean k<=K={f9_low:.3f} | k>K={f9_high:.3f} -> overfit refuted (f9 holds/rises).")
print(f"NB: final K fixed in 6a.3d (2D K x config sweep under the gap constraint). Provisional K_auto={K_auto}.")


### Block 6a.3c: Overfit diagnostic (hyperparameter grid)

In [ ]:
# Overfit diagnostic: hyperparameter grid, gap = AP_train - AP_oof. Shows that at a representative K
# (the provisional K_auto from the plateau), essentially EVERY config overfits (AP_train -> 1.0, large
# gap) -> motivates the 2D K x config sweep. Guard: reload m2_hpgrid_diag.csv.
from tqdm import tqdm
GRIDC_CSV = os.path.join(PROCESSED, 'm2_combined_hpgrid_diag.csv')
K_DIAG = K_auto
FEATS_DIAG = dedup_list[:K_DIAG]
y8, y9 = d8.label.values, f9.label.values

def diag_one(cfg, feats):
    """OOF AP / train AP / gap / fold9 AP for one config on a given feature subset (native folds)."""
    X8=d8[feats]; X9=f9[feats]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y8[trm].sum()==0 or y8[vam].sum()==0: continue
        spw=(y8[trm]==0).sum()/max((y8[trm]==1).sum(),1)
        oof[vam]=make_xgb(spw,**cfg).fit(X8[trm],y8[trm]).predict_proba(X8[vam])[:,1]
    m=~np.isnan(oof)
    ap_oof=average_precision_score(y8[m],oof[m]); auc_oof=roc_auc_score(y8[m],oof[m])
    mdl=make_xgb(**cfg).fit(X8,y8)
    ap_tr=average_precision_score(y8, mdl.predict_proba(X8)[:,1])
    ap_f9=average_precision_score(y9, mdl.predict_proba(X9)[:,1]) if y9.sum()>0 else np.nan
    return ap_oof,auc_oof,ap_tr,ap_f9

if os.path.exists(GRIDC_CSV):
    G=pd.read_csv(GRIDC_CSV); print(f"[guard] {GRIDC_CSV} reloaded ({len(G)} configs).")
else:
    GRID=[dict(max_depth=d,learning_rate=lr,n_estimators=ne,min_child_weight=mcw)
          for d in (2,3,4,5,6) for lr in (0.05,0.1) for ne in (100,200) for mcw in (1,5)]
    rows=[]
    for g in tqdm(GRID, desc='Diagnostic grid', unit='cfg'):
        ao,uo,at,af=diag_one(g, FEATS_DIAG)
        rows.append({**g,'AP_oof':ao,'AUC_oof':uo,'AP_train':at,'gap':at-ao,'AP_f9':af})
    G=pd.DataFrame(rows); G.to_csv(GRIDC_CSV,index=False)

print(f"Top 8 by AP_oof at K={K_DIAG} - note the gap explodes (AP_train -> 1.0) from ne200 on:")
print(G.sort_values('AP_oof',ascending=False).head(8)[
    ['max_depth','learning_rate','n_estimators','min_child_weight','AP_oof','AP_train','gap','AP_f9']].to_string(index=False))
print(f"\nMedian gap over the grid: {G.gap.median():.3f} -> generalized overfit. Lever = learning rate (see 6a.3d).")


### Block 6a.3d: 2D K x config grid + parsimony selection (1-SE rule)

In [ ]:
# Overfit-gap diagnostic + 2D K x config grid -> FINAL K and config.
# UNIFIED selection rule (identical to M1 05a/b/c): the native-fold (anti-shuffle) OOF AP is the honest
# judge -> MAXIMIZE OOF AP among depth-2 configs under an absolute gap cap <= 0.30. K AND config both
# come from this grid (the AP-vs-k plateau only provides a provisional K_auto that seeds the K grid).
# Selection NEVER uses the held-out fold 9 (fold-9 AP is shown as a diagnostic only).
from tqdm import tqdm
GRID2D_CSV = os.path.join(PROCESSED, 'm2_combined_grid2d.csv')   # NEW name (harmonized grid; no stale-cache clash)
KS = sorted(set([max(5, K_auto//2), K_auto, min(len(dedup_list), int(K_auto*1.5)), len(dedup_list)]))
CFGS = {'d2_lr02_ne300': dict(max_depth=2, learning_rate=0.02, n_estimators=300, min_child_weight=3),
        'd2_lr03_ne200': dict(max_depth=2, learning_rate=0.03, n_estimators=200, min_child_weight=3),
        'd2_lr03_ne300': dict(max_depth=2, learning_rate=0.03, n_estimators=300, min_child_weight=3),
        'd2_lr05_ne100': dict(max_depth=2, learning_rate=0.05, n_estimators=100, min_child_weight=3),
        'd2_lr05_ne200': dict(max_depth=2, learning_rate=0.05, n_estimators=200, min_child_weight=3),
        'd3_lr03_ne200': dict(max_depth=3, learning_rate=0.03, n_estimators=200, min_child_weight=3)}  # depth-3 = diagnostic only

if os.path.exists(GRID2D_CSV):
    G2=pd.read_csv(GRID2D_CSV); print(f"[guard] {GRID2D_CSV} reloaded ({len(G2)} combos).")
else:
    rows=[]
    for k in tqdm(KS, desc='K x config', unit='K'):
        for cn,c in CFGS.items():
            ao,uo,at,af=diag_one(c, dedup_list[:k])
            rows.append({'K':k,'cfg':cn,'AP_oof':ao,'AP_train':at,'gap':at-ao,'AP_f9':af})
    G2=pd.DataFrame(rows); G2.to_csv(GRID2D_CSV,index=False)

# Heatmaps: gap (overfit) + fold9 AP (diagnostic only)
fig,ax=plt.subplots(1,2,figsize=(13,5))
for a,(mt,ti,cm) in zip(ax,[('gap','gap train-OOF (higher = OVERFIT)','Reds'),
                            ('AP_f9','AP fold 9 (diagnostic only, NOT used for selection)','viridis')]):
    P=G2.pivot(index='K',columns='cfg',values=mt); im=a.imshow(P.values,aspect='auto',cmap=cm)
    a.set_xticks(range(len(P.columns))); a.set_xticklabels(P.columns,rotation=30,ha='right',fontsize=8)
    a.set_yticks(range(len(P.index))); a.set_yticklabels([f'K={k}' for k in P.index]); a.set_title(ti); plt.colorbar(im,ax=a)
    for ii in range(P.shape[0]):
        for jj in range(P.shape[1]):
            v=P.values[ii,jj]
            a.text(jj,ii,f'{v:.2f}',ha='center',va='center',fontsize=7,color='white' if (mt=='gap' and v>0.35) else 'black')
plt.suptitle('M2 combined - K x config sweep (selection on OOF AP among depth-2, gap<=0.30)'); plt.tight_layout()
plt.savefig(os.path.join(FIG,'m2_combined_grid2d.png'),dpi=130,bbox_inches='tight'); plt.show()

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(G2.sort_values('AP_oof', ascending=False).head(10).to_string(index=False))

# Unified, non-arbitrary selection (1-SE / bootstrap-IC rule, same principle as K_auto):
# among depth-2 configs with gap <= 0.30, take the best OOF AP, bootstrap ITS OOF AP to get the IC95
# lower bound L*, then keep the SMALLEST K whose OOF AP >= L* (statistically tied with the best). The
# "tolerance" is the data's own bootstrap noise (115 WPW) -- no magic number. Selection never uses fold 9.
GAP_CAP = 0.30
d2g = G2[G2.cfg.str.startswith('d2')]
gap_cap = max(GAP_CAP, float(d2g.gap.min()) + 0.05)   # 0.30 if achievable; else least-overfit depth-2 + margin (mono-dataset overfits everywhere)
cand = d2g[d2g.gap <= gap_cap].copy()
if len(cand) == 0: cand = G2[G2.gap <= gap_cap].copy()
top = cand.sort_values('AP_oof', ascending=False).iloc[0]
# bootstrap the OOF AP of the best config -> IC95 lower bound (reuses the 6a.3 bootstrap machinery)
oof_b = oof_xgb(dedup_list[:int(top.K)], **CFGS[top.cfg]); mb = ~np.isnan(oof_b)
ap_med_b, ic_lo_best, ic_hi_b = ap_boot(y[mb], oof_b[mb],
                                        make_boot_idx(int(y[mb].sum()), int((y[mb]==0).sum())))
tied = cand[cand.AP_oof >= ic_lo_best].copy()
best = tied.sort_values(['K','AP_oof'], ascending=[True, False]).iloc[0]   # smallest K, then best OOF
K = int(best.K); FEATURES_K = dedup_list[:K]
CFG = {**CFGS[best.cfg], 'subsample':0.8, 'colsample_bytree':0.8, 'reg_lambda':2.0}
print(f"\n[selection] 1-SE rule (gap_cap={gap_cap:.2f}): best depth-2 OOF AP={top.AP_oof:.3f} (K={int(top.K)} {top.cfg}), "
      f"bootstrap IC_lo={ic_lo_best:.3f}")
print(f"  {len(tied)} depth-2 configs statistically tied (OOF AP >= {ic_lo_best:.3f}) -> smallest K among them:")
print(f">>> FINAL CONFIG: K={K}, cfg={best.cfg} (gap {best.gap:.3f}, AP_oof {best.AP_oof:.3f}) "
      f"- parsimonious, never uses fold 9")


### Block 6a.4: Final model + standardized evaluation

In [ ]:
# Final M2 model + standardized evaluation. Raw model (folds 1-8) for threshold/SHAP/freeze; OOF on
# native folds for the F1-max threshold + Platt calibration (anti-shuffle); multi-seed stability;
# false-negative profile + extraction_failed test (interpretation). Evaluation goes through
# src/evaluation.evaluate_standard -- protocol identical for all models (F1-max threshold from OOF
# applied to fold 9, confusion, P/R/F1/spec/NPV, bootstrap IC95).
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression as _LR
from evaluation import evaluate_standard, f1max_threshold

y8, y9 = d8.label.values, f9.label.values
X8, X9 = d8[FEATURES_K], f9[FEATURES_K]

def make_final(**kw):
    p=dict(**CFG, scale_pos_weight=spw8, eval_metric='aucpr', tree_method='hist',
           random_state=42, n_jobs=10); p.update(kw); return XGBClassifier(**p)

# (1) Raw model (used for threshold/SHAP and freeze)
xgb_raw = make_final().fit(X8, y8)
score9_raw = xgb_raw.predict_proba(X9)[:,1]
ap_train   = average_precision_score(y8, xgb_raw.predict_proba(X8)[:,1])

# (2) OOF scores on native folds 1-8 (for F1-max threshold + anti-shuffle calibration)
folds8 = d8.fold.values
oof_raw = np.full(len(d8), np.nan)
for h in sorted(np.unique(folds8)):
    trm, vam = folds8!=h, folds8==h
    if y8[trm].sum()==0 or y8[vam].sum()==0: continue
    spw=(y8[trm]==0).sum()/max((y8[trm]==1).sum(),1)
    oof_raw[vam]=make_final(scale_pos_weight=spw).fit(X8[trm],y8[trm]).predict_proba(X8[vam])[:,1]
mask_oof = ~np.isnan(oof_raw)

# (3) Platt calibration on native OOF folds (anti-shuffle, consistent with M6/M1)
platt=_LR(max_iter=2000).fit(oof_raw[mask_oof].reshape(-1,1), y8[mask_oof])
def calibrate(s): return platt.predict_proba(np.asarray(s).reshape(-1,1))[:,1]
score9_cal = calibrate(score9_raw)

# (4) Multi-seed stability (fold 9)
aps=[]; aucs=[]
for s in range(10):
    mdl=make_final(random_state=s).fit(X8,y8); p=mdl.predict_proba(X9)[:,1]
    aps.append(average_precision_score(y9,p)); aucs.append(roc_auc_score(y9,p))
aps=np.array(aps); aucs=np.array(aucs)
MULTISEED=dict(AP_mean=float(aps.mean()),AP_std=float(aps.std()),
               AUC_mean=float(aucs.mean()),AUC_std=float(aucs.std()))
print(f"Multi-seed (10) fold9: AP {aps.mean():.3f}+/-{aps.std():.3f} | AUC {aucs.mean():.3f}+/-{aucs.std():.3f}")

# (5) False-negative profile + extraction_failed test (interpretation, outside the standard)
THR_tmp = f1max_threshold(y8[mask_oof], oof_raw[mask_oof])
f9w=f9[f9.label==1].copy(); f9w['score']=score9_raw[f9.label.values==1]; f9w['miss']=(f9w.score<THR_tmp)
print(f"False negatives fold9 ({int(f9w.miss.sum())}/{len(f9w)} WPW missed) "
      f"| by source {f9w[f9w.miss].source.value_counts().to_dict()} "
      f"| mean score missed {f9w[f9w.miss].score.mean():.3f} vs detected {f9w[~f9w.miss].score.mean():.3f}")
ef=df[df.fold.between(1,9)]
ef_w=ef[ef.label==1].extraction_failed.mean(); ef_n=ef[ef.label==0].extraction_failed.mean()
print(f"extraction_failed: WPW {ef_w*100:.2f}% vs non-WPW {ef_n*100:.2f}% "
      f"-> {'potential signal' if (ef_w>2*ef_n and ef_w>0.02) else 'not an exploitable signal (M2 does not delineate)'}")

# (6) STANDARDIZED EVALUATION (metrics + figure + JSON) -- identical for every model
M2_METRICS = evaluate_standard(
    name='M2_combined',
    y_oof=y8[mask_oof], score_oof=oof_raw[mask_oof],     # F1-max threshold computed here
    y_test=y9,          score_test=score9_raw,            # applied to fold 9
    figures_dir=FIG, metrics_dir=METRICS,
    score_test_calibrated=score9_cal, ap_train=ap_train,
    multiseed=MULTISEED,
    extra={'K':K,'xgb_params':{**CFG},
           'fn_by_source':f9w[f9w.miss].source.value_counts().to_dict(),
           'extraction_failed_wpw':float(ef_w),'extraction_failed_non':float(ef_n)})


### Block 6a.5: Freeze

In [ ]:
# Freeze M2 (raw + Platt calibrator, OOF, reference scores for the Flask percentile, features, config).
# Overwrites the canonical M2_statistical artifacts. Fold 10 never touched.
import joblib
from datetime import datetime
M2DIR = os.path.join(ROOT,'models','M2_statistical'); os.makedirs(M2DIR,exist_ok=True)

AP_oof  = float(average_precision_score(y8[mask_oof], oof_raw[mask_oof]))
AUC_oof = float(roc_auc_score(y8[mask_oof], oof_raw[mask_oof]))
ref_scores = np.sort(oof_raw[mask_oof])   # reference distribution for the Flask percentile

joblib.dump(xgb_raw, os.path.join(M2DIR,'m2_xgb_raw.joblib'))
joblib.dump(platt,   os.path.join(M2DIR,'m2_platt_calibrator.joblib'))
np.save(os.path.join(M2DIR,'m2_oof_folds1_8.npy'), oof_raw)
np.save(os.path.join(M2DIR,'m2_ref_scores_folds1_8.npy'), ref_scores)
pd.Series(FEATURES_K,name='feature').to_csv(os.path.join(M2DIR,'m2_selected_features.csv'),index=False)

config={
 "model":"M2_statistical_combined",
 "description":"Global statistical representation of the ECG, 100% delineation-free and peak-detection-free (stats/FFT/autocorr/Hjorth/entropy over 12 leads).",
 "date_frozen":datetime.now().strftime("%Y-%m-%d %H:%M"),
 "filter":"butterworth order 4, 0.5-40 Hz, zero-phase (frozen in 06e)",
 "n_features":{"pool":int(len(ALL_FEATS)),"gate":int(res.gate.sum()),"dedup":int(len(dedup_list)),"K":int(K)},
 "features":FEATURES_K,
 "xgb_params":{**CFG,"scale_pos_weight":float(spw8),"eval_metric":"aucpr"},
 "selection":"gate |d|>0.3 AND p_FDR<0.05 (BH) AND IC95 bootstrap excl 0 AND cross-dataset (PTB AND Ningbo); Spearman>0.9 dedup; K & config by 2D K x config grid = max OOF AP among depth-2 under absolute gap<=0.30 (unified rule, never selects on fold 9)",
 "metrics_standard":M2_METRICS,
 "AP_oof_folds1_8":AP_oof,"AUC_oof_folds1_8":AUC_oof,
 "algo_justification":"XGBoost: native NaN handling (central to the extraction-failure hypothesis) + inter-model consistency. No RF/LR baseline (architectural choice).",
 "calibration":"Platt (sigmoid) fit on native OOF folds 1-8 (anti-shuffle, consistent with M6). m2_platt_calibrator.joblib",
 "notes":"F1-max threshold computed on OOF folds 1-8, applied to fold 9. Flask display = percentile of the raw score in ref_scores (folds 1-8). Fold 10 NEVER touched.",
 "files":{"raw":"m2_xgb_raw.joblib","calibrator":"m2_platt_calibrator.joblib",
   "oof":"m2_oof_folds1_8.npy","ref_scores":"m2_ref_scores_folds1_8.npy",
   "features":"m2_selected_features.csv","metrics":"reports/metrics/M2_combined_metrics.json"}
}
with open(os.path.join(M2DIR,'m2_final_config.json'),'w',encoding='utf-8') as f:
    json.dump(config,f,ensure_ascii=False,indent=2)
print(f"=== M2 FROZEN -> {M2DIR} ===")
print(f"  AP fold9 {M2_METRICS['AP']:.3f} IC95[{M2_METRICS['AP_IC95'][0]:.2f},{M2_METRICS['AP_IC95'][1]:.2f}] | "
      f"AUC {M2_METRICS['AUC']:.3f} | F1 {M2_METRICS['confusion_at_threshold']['f1']:.3f} | K={K}")
print("  Fold 10 never touched. M2 combined (06a) done.")
